In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True # This bypasses the OSError

import tensorflow as tf
# ... rest of your code ...

# 1. Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# 2. Data Loading (Points to parent folders only)
train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, zoom_range=0.2)
val_gen = ImageDataGenerator(rescale=1./255)

train_path = r"C:\Users\JAN\Downloads\beef_data\dataset\train"
val_path = r"C:\Users\JAN\Downloads\beef_data\dataset\val"

# Use a dot '.' to mean "look right here in this folder"
#train_path = "./dataset/train" 
#val_path = "./dataset/val"

train_data = train_gen.flow_from_directory(train_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical")
val_data = val_gen.flow_from_directory(val_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical")

# 3. Model Architecture (MobileNetV2 Transfer Learning)
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False 

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax') # Will be 2: Beef and Others
])

# 4. Compile and Train
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, validation_data=val_data, epochs=10)

# 5. Save the "Brain"
model.save("food_classifier_model.h5")
print("Model Saved Successfully!")

Found 1100 images belonging to 2 classes.
Found 277 images belonging to 2 classes.
Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 28s 340ms/step - accuracy: 0.9818 - loss: 0.0554 - val_accuracy: 1.0000 - val_loss: 0.0065
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 21s 306ms/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 22s 312ms/step - accuracy: 1.0000 - loss: 3.6857e-04 - val_accuracy: 0.9964 - val_loss: 0.0047
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 22s 320ms/step - accuracy: 0.9991 - loss: 0.0033 - val_accuracy: 0.9964 - val_loss: 0.0075
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 21s 308ms/step - accuracy: 0.9991 - loss: 0.0023 - val_accuracy: 0.9928 - val_loss: 0.0132
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 21s 309ms/step - accuracy: 0.9991 - loss: 0.0023 - val_accuracy: 0.9856 - val_loss: 0.0329
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 22s 313ms/step - accuracy: 0.9991 - loss: 0.0031 - val_accuracy: 0.9928 - val_loss: 0.0200
Epoch 8/10

Model Saved Successfully!


In [13]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import os

# 1. Load the "Brain" you just trained
model = tf.keras.models.load_model('food_classifier_model.h5')

# 2. Configuration
IMG_SIZE = (224, 224)
THRESHOLD = 0.85  # 85% confidence required

def predict_food(img_path):
    # Load and preprocess the image
    #img = image.load_img(img_path, target_size=IMG_SIZE)
    img = image.load_img(img_path, target_size=(224, 224), interpolation='bilinear')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Make it a "batch" of 1
    img_array /= 255.0  # Normalize like we did in training

    # Run Prediction
    predictions = model.predict(img_array, verbose=0)[0]
    
    # Assuming folder order was: [0] beef, [1] others
    # (Keras flow_from_directory uses alphabetical order)
    beef_confidence = predictions[0]
    others_confidence = predictions[1]

    print(f"\n--- Results for: {os.path.basename(img_path)} ---")
    print(f"Beef Confidence: {beef_confidence:.2%}")
    print(f"Others Confidence: {others_confidence:.2%}")

    # 3. Decision Logic for your Product
    if beef_confidence >= THRESHOLD:
        return "RESULT: [BEEF] - Proceeding to Freshness Analysis (Gas Sensors)."
    else:
        return "RESULT: [NOT AVAILABLE] - Food item not recognized or incorrect type."

# 4. TEST IT: Pick one image from your val/beef folder
test_image = r"C:\Users\JAN\Downloads\beef_data\dataset\val\beef\additional_beef.jpg" 

# Check if file exists before running
if os.path.exists(test_image):
    print(predict_food(test_image))
else:
    print("Error: Please replace 'ANY_IMAGE_NAME.jpg' with a real filename from your folder.")



--- Results for: additional_beef.jpg ---
Beef Confidence: 8.16%
Others Confidence: 91.84%
RESULT: [NOT AVAILABLE] - Food item not recognized or incorrect type.


In [1]:
import tensorflow as tf

# Load the brain
model = tf.keras.models.load_model('food_classifier_model.h5')

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the converted model
with open('food_classifier_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ Success! Your model is now an Edge-Ready .tflite file.")

INFO:tensorflow:Assets written to: C:\Users\JAN\AppData\Local\Temp\tmpxwfjq1r5\assets


INFO:tensorflow:Assets written to: C:\Users\JAN\AppData\Local\Temp\tmpxwfjq1r5\assets


Saved artifact at 'C:\Users\JAN\AppData\Local\Temp\tmpxwfjq1r5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2670480541120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480552912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480555024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480548336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480550624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480551328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480656160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480654752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480656336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2670480657568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  267048

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# 1. Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="food_classifier_model.tflite")
interpreter.allocate_tensors()

# 2. Get input and output details (tells the code where the "doors" are)
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_tflite(img_path):
    # 3. Preprocess the image exactly like before
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)
    img_array /= 255.0

    # 4. Set the input tensor
    interpreter.set_tensor(input_details[0]['index'], img_array)

    # 5. Run the "Inference" (the actual calculation)
    interpreter.invoke()

    # 6. Get the result
    predictions = interpreter.get_tensor(output_details[0]['index'])[0]
    
    beef_conf = predictions[0]
    return beef_conf

# Test it
test_path = r"C:\Users\JAN\Downloads\beef_data\dataset\val\beef\20210917_B01_10_seg.jpg"
result = predict_tflite(test_path)

THRESHOLD = 0.60 # Your new optimized threshold

print(f"\n--- TFLite Test Results ---")
print(f"Beef Confidence: {result:.2%}")

if result >= THRESHOLD:
    print("✅ RESULT: [BEEF DETECTED] - Proceeding to sensor analysis.")
else:
    print("❌ RESULT: [NOT RECOGNIZED] - System standby.")


--- TFLite Test Results ---
Beef Confidence: 99.78%
✅ RESULT: [BEEF DETECTED] - Proceeding to sensor analysis.


C:\Users\JAN\anaconda3\envs\ml_env\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
